# HR Salary Prediction System: EDA, Data Cleaning & Machine Learning Pipeline

**Objective:** Build a robust compensation benchmarking model to predict HR professional salaries based on Age and Years of Experience.

### Success Criteria
- R² Score >= 0.50
- MAE (Mean Absolute Error) < 4,000 INR

## 1. Dataset Description
- **Source**: Historical HR payroll database (synthetic).
- **Number of Rows**: 200,000
- **Number of Columns**: 3
- **Target Column**: `Target_Salary` (Continuous Monthly/Annual Salary in INR)
- **Feature Columns**: `Age` and `Years_of_Experience`

## 2. Environment Setup & Loading Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
df_raw = pd.read_csv('data/raw/hr_salary_data.csv')
print('Dataset Shape:', df_raw.shape)
df_raw.head()

## 3. Dedicated Exploratory Data Analysis (EDA)

We systematically examine data shapes, schemas, distributions, outliers, and correlations.

In [ ]:
# Dataset Shape and Basic Info
print('Dataset Dimensions:', df_raw.shape)
df_raw.info()

# Descriptive Statistics
df_raw.describe()

In [ ]:
# Missing Values Check
print('Missing Values per column:')
print(df_raw.isnull().sum())

# Duplicates Check
print('\nDuplicate records count:', df_raw.duplicated().sum())

In [ ]:
# Correlation Matrix Heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(df_raw.corr(), annot=True, cmap='coolwarm', fmt='.3f', vmin=-1, vmax=1)
plt.title('Correlation Matrix Heatmap (Raw Data)')
plt.show()

In [ ]:
# Outlier Detection Boxplots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.boxplot(ax=axes[0], y=df_raw['Age'], color='skyblue')
axes[0].set_title('Age Boxplot')

sns.boxplot(ax=axes[1], y=df_raw['Years_of_Experience'], color='lightgreen')
axes[1].set_title('Experience Boxplot')

sns.boxplot(ax=axes[2], y=df_raw['Target_Salary'], color='salmon')
axes[2].set_title('Salary Boxplot')
plt.show()

In [ ]:
# Distribution Histograms & KDEs
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(ax=axes[0], data=df_raw, x='Age', kde=True, color='skyblue')
axes[0].set_title('Age Distribution')

sns.histplot(ax=axes[1], data=df_raw, x='Years_of_Experience', kde=True, color='lightgreen')
axes[1].set_title('Experience Distribution')

sns.histplot(ax=axes[2], data=df_raw, x='Target_Salary', kde=True, color='salmon')
axes[2].set_title('Salary Distribution')
plt.show()

In [ ]:
# Pair Plot on a representative sample of 2,000 rows to speed up rendering
print('Plotting Pair Plot (Scatter Plot Matrix)...')
sns.pairplot(df_raw.sample(2000, random_state=42), hue='Age', palette='viridis')
plt.show()

## 4. Data Cleaning & Validation

We document and execute standard data quality steps:
1. **Missing values**: Imputed or dropped if found.
2. **Duplicates**: Filtered to ensure unique records.
3. **Type validation**: Cast columns to logical types (`int64` and `float64`).
4. **Outliers (IQR)**: Checked target spread and handled negative values.
5. **Physical inconsistencies**: Removed cases where experience exceeded Age - 18 working age limit.

In [ ]:
print('--- Starting Cleaning Pipeline ---')
# Deduplicate and handle nulls
df_clean = df_raw.dropna().drop_duplicates().reset_index(drop=True)
print(f'Size after null/duplicate checks: {len(df_clean)}')

# Type conversion
df_clean['Age'] = df_clean['Age'].astype('int64')
df_clean['Years_of_Experience'] = df_clean['Years_of_Experience'].astype('int64')
df_clean['Target_Salary'] = df_clean['Target_Salary'].astype('float64')

# Negative values filter
df_clean = df_clean[df_clean['Target_Salary'] >= 0].reset_index(drop=True)
print(f'Size after negative salary filter: {len(df_clean)}')

# Age-Experience consistency filter
df_logical = df_clean[df_clean['Years_of_Experience'] <= (df_clean['Age'] - 18)].reset_index(drop=True)
print(f'Size after consistency rules (Final clean dataset): {len(df_logical)}')

df_logical.to_csv('data/processed/hr_salary_cleaned.csv', index=False)

## 5. Feature Engineering

In a standard production environment, we would use:
- **Feature Scaling**: Scaled features to prevent high-magnitude features from dominating models.
- **Feature Selection**: Correlation heatmap shows low multicollinearity between Age and Experience.
- **Polynomial / Interaction Features**: Optional generation of interactions like `Age_x_Experience` to capture non-linear synergies.

Let's split our dataset and fit scaling.

In [ ]:
X = df_logical[['Age', 'Years_of_Experience']]
y = df_logical['Target_Salary']

# Train-test split (Strictly before scaling to avoid data leakage)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns)
print('Features scaled successfully.')

## 6. Model Selection & Cross Validation

We train and cross-validate 6 regression models:
1. Linear Regression
2. Ridge Regression
3. Lasso Regression
4. Decision Tree
5. Random Forest
6. XGBoost Regressor

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=1.0),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBoost Regressor': XGBRegressor(n_estimators=100, random_state=42, learning_rate=0.1, max_depth=6)
}

cv_results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2', n_jobs=-1)
    cv_results[name] = {'cv_mean': np.mean(scores), 'cv_std': np.std(scores)}
    print(f'Trained {name:20s} | Mean CV R2: {np.mean(scores):.4f}')

In [ ]:
results = []
for name, model in models.items():
    preds = model.predict(X_test_scaled)
    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, preds)
    results.append({
        'Model': name,
        'MAE': mae,
        'RMSE': rmse,
        'R2 Score': r2,
        'CV R2 Mean': cv_results[name]['cv_mean'],
        'CV R2 Std': cv_results[name]['cv_std']
    })

metrics_df = pd.DataFrame(results)
metrics_df.sort_values(by='R2 Score', ascending=False)

## 7. Hyperparameter Tuning (Random Forest)

We perform hyperparameter optimization via `GridSearchCV` and `RandomizedSearchCV` on a sample of 10,000 rows.

In [ ]:
# Sample data for speed
indices = np.random.choice(len(X_train_scaled), size=10000, replace=False)
X_sample = X_train_scaled.iloc[indices]
y_sample = y_train.iloc[indices]

param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

rf = RandomForestRegressor(random_state=42)

grid_search = GridSearchCV(rf, param_grid, cv=3, scoring='r2', n_jobs=-1)
grid_search.fit(X_sample, y_sample)

random_search = RandomizedSearchCV(rf, param_grid, n_iter=10, cv=3, scoring='r2', random_state=42, n_jobs=-1)
random_search.fit(X_sample, y_sample)

print('GridSearchCV Best Params:', grid_search.best_params_)
print('GridSearchCV Best CV R2:', grid_search.best_score_)
print('RandomizedSearchCV Best Params:', random_search.best_params_)
print('RandomizedSearchCV Best CV R2:', random_search.best_score_)

In [ ]:
best_params = grid_search.best_params_
tuned_rf = RandomForestRegressor(**best_params, random_state=42)
tuned_rf.fit(X_train_scaled, y_train)

# Compare on full Test set
preds_base = models['Random Forest'].predict(X_test_scaled)
preds_tuned = tuned_rf.predict(X_test_scaled)

comp_tuning = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'R2 Score'],
    'Before Tuning (Base RF)': [
        mean_absolute_error(y_test, preds_base),
        np.sqrt(mean_squared_error(y_test, preds_base)),
        r2_score(y_test, preds_base)
    ],
    'After Tuning (Tuned RF)': [
        mean_absolute_error(y_test, preds_tuned),
        np.sqrt(mean_squared_error(y_test, preds_tuned)),
        r2_score(y_test, preds_tuned)
    ]
})
comp_tuning

## 8. Model Interpretation & Diagnostic Plots

We check feature importances, prediction errors, residual plots, and learning curves.

In [ ]:
# Feature Importance Chart
importances = tuned_rf.feature_importances_
df_imp = pd.DataFrame({'Feature': X.columns, 'Importance': importances}).sort_values(by='Importance', ascending=False)
sns.barplot(x='Importance', y='Feature', data=df_imp, palette='viridis')
plt.title('Feature Importances in Salary Prediction')
plt.show()

In [ ]:
# 1. Actual vs Predicted
best_model_name = 'Lasso Regression'
best_model = models[best_model_name]
best_preds = best_model.predict(X_test_scaled)

plt.figure(figsize=(10, 5))
sns.scatterplot(x=y_test, y=best_preds, alpha=0.4, color='teal')
min_v, max_v = min(y_test.min(), best_preds.min()), max(y_test.max(), best_preds.max())
plt.plot([min_v, max_v], [min_v, max_v], 'r--', lw=2, label='Ideal Fit')
plt.title('Actual vs Predicted Salaries (Prediction Error)')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.legend()
plt.show()

In [ ]:
# 2. Residual Plot
residuals = y_test - best_preds
plt.figure(figsize=(10, 5))
sns.scatterplot(x=best_preds, y=residuals, alpha=0.4, color='purple')
plt.axhline(0, color='red', linestyle='--', lw=2)
plt.title('Residuals vs Predicted')
plt.xlabel('Predicted')
plt.ylabel('Residuals')
plt.show()

In [ ]:
# 3. Learning Curve for Best Model
train_sizes, train_scores, val_scores = learning_curve(
    best_model, X_train_scaled, y_train, cv=3, scoring='r2', n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 5), random_state=42
)
train_mean = np.mean(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)
plt.plot(train_sizes, train_mean, 'o-', color='blue', label='Training score')
plt.plot(train_sizes, val_mean, 'o-', color='green', label='Cross-validation score')
plt.title('Learning Curve')
plt.xlabel('Training size')
plt.ylabel('R2 Score')
plt.legend()
plt.show()

In [ ]:
# Error analysis by experience categories
df_errors = pd.DataFrame({
    'Experience': X_test['Years_of_Experience'],
    'Absolute Error': np.abs(residuals)
})
df_errors['Exp_Group'] = pd.cut(df_errors['Experience'], bins=[-1, 5, 15, 30, 40], labels=['0-5 Yrs', '5-15 Yrs', '15-30 Yrs', '30+ Yrs'])
print('Mean Absolute Error by Experience Group:')
print(df_errors.groupby('Exp_Group', observed=False)['Absolute Error'].mean())

## 9. Business Insights & Final Summary

- **Years of Experience** is the core pricing engine for payroll benchmarking, holding ~97% predictive importance.
- **Age** has very little direct influence once experience is controlled.
- Models perform closely because the payroll structure in this synthetic dataset is highly linear with uniform noise.